## Tool definition

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

<img src="./resources/tools.png" width="400" style="display:block; margin-left:0;">

## 工具 
#### 这个图可以看到大模型可以通过工具来扩大自己的能力，通过理理解内用户的问题, 产生调用工具和参数，调用工具，最后汇总结结果


### 我们先定一些基本的工具， 就是算术计算的函数， 例如加减乘除。

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [ ]:
tool1.invoke({"x": 467})

## 把工具赋能给agent

####  在agent 的基础之上， 增加工具，这样， agent 可以使用这些工具来扩展自己的能力。

In [5]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# model = init_chat_model("deepseek-chat")
# Alternative: local model via Ollama
model = init_chat_model("qwen2.5:14b", model_provider="ollama")

agent = create_agent(
    model=model,
    tools=[tool1],
    system_prompt="你是一个算术巫师。使用你的工具来计算任何数字的平方根和平方。"
)

In [6]:
from langchain.messages import HumanMessage

question = HumanMessage(content="467的平方根是多少？")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

467的平方根约为21.61。


Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError(\'HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Max retries exceeded with url: /info (Caused by NameResolutionError("HTTPSConnection(host=\\\'api.smith.langchain.com\\\', port=443): Failed to resolve \\\'api.smith.langchain.com\\\' ([Errno 8] nodename nor servname provided, or not known)"))\'))\nContent-Length: None\nAPI Key: lsv2_********************************************aa')


#### 注意agent 并不能直接使用工具，它是知道调用哪个工具，，并要把工具的参数赋给传给工具，工具去执行之后再交给agent 汇总。

In [7]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='467的平方根是多少？', additional_kwargs={}, response_metadata={}, id='06e32c5f-c753-4147-b660-dd2acb4295fd'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:14b', 'created_at': '2026-05-28T06:41:37.537616Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1444532667, 'load_duration': 74998417, 'prompt_eval_count': 157, 'prompt_eval_duration': 572400167, 'eval_count': 22, 'eval_duration': 780771251, 'logprobs': None, 'model_name': 'qwen2.5:14b', 'model_provider': 'ollama'}, id='lc_run--019e6d51-295a-7221-bbc9-7a020c1b2871-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'a7882832-4c4c-4779-b986-b8676afef387', 'type': 'tool_call'}], usage_metadata={'input_tokens': 157, 'output_tokens': 22, 'total_tokens': 179}),
 ToolMessage(content='21.61018278497431', name='square_root', id='7c2522e2-577c-40c4-8ad2-8448d2afc868', tool_call_id='a7882832-4c4c-4779-b986-b8676afef387'),
 AIMessage(content='467的平方根约为21.61。', additiona

In [8]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'a7882832-4c4c-4779-b986-b8676afef387', 'type': 'tool_call'}]


Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/batch (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: 20277
API Key: lsv2_********************************************aa
post: trace=019e6d51-28f0-7b31-affb-0f8d0f801482,id=019e6d51-28f0-7b31-affb-0f8d0f801482; trace=019e6d51-28f0-7b31-affb-0f8d0f801482,id=019e6d51-2958-7121-95fd-1d67096c716c; trace=019e6d51-28f0-7b31-affb-0f8d0f801482,id=019e6d51-295a-7221-bbc9-7a020c1b2871; trace=019e6d51-28f0-7b31-affb-0f8d0f801482,id=019e6d51-2f03-79f2-9ea9-6277e6861f81; trace=019e6d51-28f0-7